In [1]:
import os
import math
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)


# ============================================================
# Configuration
# ============================================================

FEATURE_PATH = "elliptic_txs_features.csv"
CLASS_PATH = "elliptic_txs_classes.csv"
EDGE_PATH = "elliptic_txs_edgelist.csv"   # Loaded for completeness; not directly used here
OUTPUT_DIR = "elliptic_outputs"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# High-confidence pseudo-labeling thresholds
CONFIDENCE_THRESHOLD = 0.90
MARGIN_THRESHOLD = 0.20


# ============================================================
# Utility functions
# ============================================================

def map_label_binary(x):
    """
    Map original labels to binary:
    illicit -> 1
    licit   -> 0
    unknown -> NaN
    """
    x = str(x)
    if x == "1":
        return 1
    elif x == "2":
        return 0
    else:
        return np.nan


def save_plot(path):
    """
    Save current matplotlib figure and close it.
    """
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()


# ============================================================
# Data loading and splitting
# ============================================================

def load_and_merge_data(feature_path, class_path, edge_path=None):
    """
    Load feature/class tables and merge them on txId.
    """
    features = pd.read_csv(feature_path, header=None)
    classes = pd.read_csv(class_path)

    if edge_path is not None and os.path.exists(edge_path):
        _ = pd.read_csv(edge_path)  # Not directly used in this workflow

    num_cols = features.shape[1]
    feature_cols = ["txId", "time_step"] + [f"f_{i}" for i in range(num_cols - 2)]
    features.columns = feature_cols
    classes.columns = ["txId", "class"]

    data = features.merge(classes, on="txId", how="left")
    return data, feature_cols[2:]


def split_by_time(data, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15):
    """
    Split the dataset by unique time_step values.
    """
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-8, "Ratios must sum to 1."

    unique_ts = sorted(data["time_step"].unique())
    n_ts = len(unique_ts)

    n_train_ts = math.floor(train_ratio * n_ts)
    n_val_ts = math.floor(val_ratio * n_ts)

    train_ts = unique_ts[:n_train_ts]
    val_ts = unique_ts[n_train_ts:n_train_ts + n_val_ts]
    test_ts = unique_ts[n_train_ts + n_val_ts:]

    train_set = data[data["time_step"].isin(train_ts)].copy()
    val_set = data[data["time_step"].isin(val_ts)].copy()
    test_set = data[data["time_step"].isin(test_ts)].copy()

    return train_set, val_set, test_set, train_ts, val_ts, test_ts


# ============================================================
# Random Fourier Features + Kernel Logistic Regression
# ============================================================

class RandomFourierFeatures(nn.Module):
    """
    Random Fourier Features approximation for the RBF kernel.
    """
    def __init__(self, input_dim, output_dim=1024, gamma=1.0):
        super().__init__()
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.gamma = gamma

        W = torch.randn(input_dim, output_dim) * math.sqrt(2 * gamma)
        b = 2 * math.pi * torch.rand(output_dim)

        self.register_buffer("W", W)
        self.register_buffer("b", b)

    def forward(self, x):
        z = x @ self.W + self.b
        z = math.sqrt(2.0 / self.output_dim) * torch.cos(z)
        return z


class KernelLogisticRegression(nn.Module):
    """
    Approximate kernel logistic regression using RFF + linear layer.
    """
    def __init__(self, input_dim, rff_dim=1024, gamma=1.0):
        super().__init__()
        self.rff = RandomFourierFeatures(input_dim=input_dim, output_dim=rff_dim, gamma=gamma)
        self.linear = nn.Linear(rff_dim, 1)

    def forward(self, x):
        z = self.rff(x)
        logits = self.linear(z).squeeze(1)
        return logits


def predict_proba(model, X, batch_size=4096):
    """
    Predict probabilities for a numpy array.
    """
    model.eval()
    probs_all = []

    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.no_grad():
        for (xb,) in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            probs_all.append(probs)

    return np.concatenate(probs_all)


def train_klr_model(
    X_train,
    y_train,
    X_val=None,
    y_val=None,
    input_dim=None,
    rff_dim=1024,
    gamma=0.5,
    lr=1e-3,
    weight_decay=1e-4,
    batch_size=2048,
    epochs=30,
    patience=5,
    verbose=True
):
    """
    Train a KLR model. If validation data is provided, use early stopping by validation AUC.
    """
    model = KernelLogisticRegression(
        input_dim=input_dim,
        rff_dim=rff_dim,
        gamma=gamma
    ).to(DEVICE)

    Xtr = torch.tensor(X_train, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.float32)

    train_ds = TensorDataset(Xtr, ytr)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    pos_count = float(y_train.sum())
    neg_count = float(len(y_train) - pos_count)
    pos_weight_value = neg_count / max(pos_count, 1.0)
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {"train_loss": []}
    if X_val is not None and y_val is not None:
        history["val_auc"] = []

    best_state = copy.deepcopy(model.state_dict())
    best_metric = -np.inf
    best_epoch = 0
    wait = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for xb, yb in train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)

        avg_loss = total_loss / len(train_ds)
        history["train_loss"].append(avg_loss)

        if X_val is not None and y_val is not None:
            val_probs = predict_proba(model, X_val)
            val_auc = roc_auc_score(y_val, val_probs)
            history["val_auc"].append(val_auc)

            if verbose:
                print(
                    f"Epoch {epoch + 1:02d}/{epochs} | "
                    f"train_loss={avg_loss:.6f} | val_auc={val_auc:.6f}"
                )

            if val_auc > best_metric:
                best_metric = val_auc
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch + 1
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch + 1}")
                    break
        else:
            if verbose:
                print(f"Epoch {epoch + 1:02d}/{epochs} | train_loss={avg_loss:.6f}")
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

    model.load_state_dict(best_state)

    return {
        "model": model,
        "best_epoch": best_epoch,
        "best_metric": best_metric if X_val is not None and y_val is not None else None,
        "history": history
    }


# ============================================================
# Dataset preparation
# ============================================================

def prepare_labeled_data(df, feature_cols, label_col):
    """
    Extract X and y from a dataframe, dropping unknown labels.
    """
    y = df[label_col].map(map_label_binary).values
    X = df[feature_cols].values.astype(np.float32)

    mask = ~np.isnan(y)
    X = X[mask]
    y = y[mask].astype(np.float32)

    return X, y


def prepare_unknown_data(df, feature_cols):
    """
    Extract unknown rows and feature matrix.
    """
    unknown_df = df[df["class"].astype(str) == "unknown"].copy()
    X_unknown = unknown_df[feature_cols].values.astype(np.float32)
    return unknown_df, X_unknown


# ============================================================
# High-confidence pseudo-labeling
# ============================================================

def fill_unknown_labels_in_train(
    train_set,
    feature_cols,
    confidence_threshold=0.80,
    margin_threshold=0.15
):
    """
    Train klr_lic and klr_ill on labeled training samples,
    then use them to pseudo-label unknown samples with high confidence only.

    A sample is filled only if:
    1. max(p_licit, p_illicit) >= confidence_threshold
    2. abs(p_licit - p_illicit) >= margin_threshold

    Otherwise, it stays as 'unknown'.
    """
    train_labeled = train_set[train_set["class"].astype(str) != "unknown"].copy()

    X_train = train_labeled[feature_cols].values.astype(np.float32)
    y_text = train_labeled["class"].astype(str).values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)

    y_lic = (y_text == "2").astype(np.float32)   # licit positive
    y_ill = (y_text == "1").astype(np.float32)   # illicit positive

    print("\nTraining klr_lic ...")
    klr_lic_result = train_klr_model(
        X_train=X_train_scaled,
        y_train=y_lic,
        input_dim=X_train_scaled.shape[1],
        rff_dim=1024,
        gamma=0.5,
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=2048,
        epochs=20,
        patience=5,
        verbose=True
    )

    print("\nTraining klr_ill ...")
    klr_ill_result = train_klr_model(
        X_train=X_train_scaled,
        y_train=y_ill,
        input_dim=X_train_scaled.shape[1],
        rff_dim=1024,
        gamma=0.5,
        lr=1e-3,
        weight_decay=1e-4,
        batch_size=2048,
        epochs=20,
        patience=5,
        verbose=True
    )

    klr_lic = klr_lic_result["model"]
    klr_ill = klr_ill_result["model"]

    train_unknown, X_unknown = prepare_unknown_data(train_set, feature_cols)

    if len(train_unknown) == 0:
        train_set_filled = train_set.copy()
        train_set_filled["p_licit"] = np.nan
        train_set_filled["p_illicit"] = np.nan
        train_set_filled["pseudo_confidence"] = np.nan
        train_set_filled["pseudo_margin"] = np.nan
        train_set_filled["pred_class"] = np.nan
        train_set_filled["class_filled"] = train_set_filled["class"]
        return train_set_filled, scaler, klr_lic, klr_ill

    X_unknown_scaled = scaler.transform(X_unknown).astype(np.float32)

    p_licit = predict_proba(klr_lic, X_unknown_scaled)
    p_illicit = predict_proba(klr_ill, X_unknown_scaled)

    train_unknown["p_licit"] = p_licit
    train_unknown["p_illicit"] = p_illicit

    train_unknown["pseudo_confidence"] = np.maximum(
        train_unknown["p_licit"],
        train_unknown["p_illicit"]
    )

    train_unknown["pseudo_margin"] = np.abs(
        train_unknown["p_licit"] - train_unknown["p_illicit"]
    )

    train_unknown["pred_class_raw"] = np.where(
        train_unknown["p_licit"] >= train_unknown["p_illicit"],
        "2",
        "1"
    )

    confident_mask = (
        (train_unknown["pseudo_confidence"] >= confidence_threshold) &
        (train_unknown["pseudo_margin"] >= margin_threshold)
    )

    train_unknown["pred_class"] = np.where(
        confident_mask,
        train_unknown["pred_class_raw"],
        "unknown"
    )

    print("\nPseudo-labeling summary:")
    print(f"Unknown samples total   : {len(train_unknown)}")
    print(f"Confident pseudo-labels : {int(confident_mask.sum())}")
    print(f"Left as unknown         : {int((~confident_mask).sum())}")

    confident_df = train_unknown.loc[confident_mask].copy()
    confident_df.to_csv(
        os.path.join(OUTPUT_DIR, "confident_pseudo_labels.csv"),
        index=False
    )

    if len(confident_df) > 0:
        print("\nPseudo-label distribution among confident samples:")
        print(confident_df["pred_class"].value_counts(dropna=False))

    plt.figure(figsize=(8, 5))
    plt.hist(train_unknown["p_licit"], bins=50)
    plt.xlabel("Predicted probability of licit")
    plt.ylabel("Frequency")
    plt.title("Unknown Samples: Predicted Probability of Licit")
    save_plot(os.path.join(OUTPUT_DIR, "unknown_p_licit_hist.png"))

    plt.figure(figsize=(8, 5))
    plt.hist(train_unknown["p_illicit"], bins=50)
    plt.xlabel("Predicted probability of illicit")
    plt.ylabel("Frequency")
    plt.title("Unknown Samples: Predicted Probability of Illicit")
    save_plot(os.path.join(OUTPUT_DIR, "unknown_p_illicit_hist.png"))

    plt.figure(figsize=(8, 5))
    plt.hist(train_unknown["pseudo_confidence"], bins=50)
    plt.axvline(confidence_threshold, linestyle="--")
    plt.xlabel("Pseudo-label confidence")
    plt.ylabel("Frequency")
    plt.title("Unknown Samples: Pseudo-label Confidence")
    save_plot(os.path.join(OUTPUT_DIR, "unknown_pseudo_confidence_hist.png"))

    plt.figure(figsize=(8, 5))
    plt.hist(train_unknown["pseudo_margin"], bins=50)
    plt.axvline(margin_threshold, linestyle="--")
    plt.xlabel("Pseudo-label margin")
    plt.ylabel("Frequency")
    plt.title("Unknown Samples: Pseudo-label Margin")
    save_plot(os.path.join(OUTPUT_DIR, "unknown_pseudo_margin_hist.png"))

    fill_map = train_unknown[
        ["txId", "p_licit", "p_illicit", "pseudo_confidence", "pseudo_margin", "pred_class"]
    ].copy()

    train_set_filled = train_set.copy()
    train_set_filled = train_set_filled.merge(fill_map, on="txId", how="left")

    train_set_filled["class_filled"] = np.where(
        (train_set_filled["class"].astype(str) == "unknown") &
        (train_set_filled["pred_class"].astype(str) != "unknown"),
        train_set_filled["pred_class"],
        train_set_filled["class"].astype(str)
    )

    return train_set_filled, scaler, klr_lic, klr_ill


# ============================================================
# Hyperparameter tuning for final KLR
# ============================================================

def tune_final_klr(train_filled, val_set, feature_cols):
    """
    Train on filled train set and tune hyperparameters using val set.
    Final task: binary classification (illicit=1, licit=0).
    """
    X_train, y_train = prepare_labeled_data(train_filled, feature_cols, label_col="class_filled")
    X_val, y_val = prepare_labeled_data(val_set, feature_cols, label_col="class")

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    X_val_scaled = scaler.transform(X_val).astype(np.float32)

    param_grid = {
        "rff_dim": [512, 1024, 2048],
        "gamma": [0.1, 0.5, 1.0],
        "lr": [1e-3, 5e-4],
        "weight_decay": [1e-4, 1e-5]
    }

    search_results = []
    best_result = None
    best_params = None
    best_auc = -np.inf

    print("\nStarting hyperparameter tuning ...")

    for rff_dim in param_grid["rff_dim"]:
        for gamma in param_grid["gamma"]:
            for lr in param_grid["lr"]:
                for wd in param_grid["weight_decay"]:
                    print(
                        f"Trying params: rff_dim={rff_dim}, gamma={gamma}, "
                        f"lr={lr}, weight_decay={wd}"
                    )

                    result = train_klr_model(
                        X_train=X_train_scaled,
                        y_train=y_train,
                        X_val=X_val_scaled,
                        y_val=y_val,
                        input_dim=X_train_scaled.shape[1],
                        rff_dim=rff_dim,
                        gamma=gamma,
                        lr=lr,
                        weight_decay=wd,
                        batch_size=2048,
                        epochs=30,
                        patience=5,
                        verbose=False
                    )

                    val_auc = result["best_metric"]

                    row = {
                        "rff_dim": rff_dim,
                        "gamma": gamma,
                        "lr": lr,
                        "weight_decay": wd,
                        "best_val_auc": val_auc,
                        "best_epoch": result["best_epoch"]
                    }
                    search_results.append(row)

                    print(f"Validation AUC = {val_auc:.6f}")

                    if val_auc > best_auc:
                        best_auc = val_auc
                        best_result = result
                        best_params = row

    search_df = pd.DataFrame(search_results).sort_values("best_val_auc", ascending=False)
    search_df.to_csv(os.path.join(OUTPUT_DIR, "klr_val_tuning_results.csv"), index=False)

    plt.figure(figsize=(8, 5))
    plt.hist(search_df["best_val_auc"], bins=20)
    plt.xlabel("Validation AUC")
    plt.ylabel("Frequency")
    plt.title("Distribution of Validation AUC Across Hyperparameter Search")
    save_plot(os.path.join(OUTPUT_DIR, "val_auc_search_hist.png"))

    return {
        "best_result": best_result,
        "best_params": best_params,
        "search_df": search_df,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val
    }


# ============================================================
# Retrain final model on train + val
# ============================================================

def retrain_final_model(X_train, y_train, X_val, y_val, best_params):
    """
    Merge train and val, then retrain final model with best hyperparameters.
    """
    X_final = np.vstack([X_train, X_val])
    y_final = np.concatenate([y_train, y_val])

    scaler = StandardScaler()
    X_final_scaled = scaler.fit_transform(X_final).astype(np.float32)

    final_result = train_klr_model(
        X_train=X_final_scaled,
        y_train=y_final,
        X_val=None,
        y_val=None,
        input_dim=X_final_scaled.shape[1],
        rff_dim=int(best_params["rff_dim"]),
        gamma=float(best_params["gamma"]),
        lr=float(best_params["lr"]),
        weight_decay=float(best_params["weight_decay"]),
        batch_size=2048,
        epochs=int(best_params["best_epoch"]),
        patience=5,
        verbose=True
    )

    return {
        "model": final_result["model"],
        "scaler": scaler,
        "history": final_result["history"],
        "X_final": X_final,
        "y_final": y_final
    }


# ============================================================
# Test evaluation
# ============================================================

def evaluate_on_test(model, scaler, test_set, feature_cols, final_history=None):
    """
    Evaluate final model on test set and generate metrics/plots.
    """
    X_test, y_test = prepare_labeled_data(test_set, feature_cols, label_col="class")
    X_test_scaled = scaler.transform(X_test).astype(np.float32)

    test_probs = predict_proba(model, X_test_scaled)
    test_preds = (test_probs >= 0.5).astype(int)

    acc = accuracy_score(y_test, test_preds)
    prec = precision_score(y_test, test_preds, zero_division=0)
    rec = recall_score(y_test, test_preds, zero_division=0)
    f1 = f1_score(y_test, test_preds, zero_division=0)
    auc = roc_auc_score(y_test, test_probs)
    ap = average_precision_score(y_test, test_probs)
    cm = confusion_matrix(y_test, test_preds)

    composite_score = 0.35 * auc + 0.35 * ap + 0.15 * f1 + 0.15 * rec

    metrics_df = pd.DataFrame([{
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
        "average_precision": ap,
        "composite_score": composite_score
    }])
    metrics_df.to_csv(os.path.join(OUTPUT_DIR, "test_metrics.csv"), index=False)

    print("\n=== TEST METRICS ===")
    print(f"Accuracy     : {acc:.6f}")
    print(f"Precision    : {prec:.6f}")
    print(f"Recall       : {rec:.6f}")
    print(f"F1 Score     : {f1:.6f}")
    print(f"ROC AUC      : {auc:.6f}")
    print(f"Average Prec.: {ap:.6f}")
    print(f"Composite    : {composite_score:.6f}")

    print("\n=== CONFUSION MATRIX ===")
    print(cm)

    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test, test_preds, digits=4, target_names=["licit", "illicit"]))

    if final_history is not None and "train_loss" in final_history:
        plt.figure(figsize=(8, 5))
        plt.plot(final_history["train_loss"])
        plt.xlabel("Epoch")
        plt.ylabel("Train Loss")
        plt.title("Final Model Training Loss")
        save_plot(os.path.join(OUTPUT_DIR, "final_train_loss.png"))

    plt.figure(figsize=(8, 5))
    plt.hist(test_probs, bins=50)
    plt.xlabel("Predicted probability of illicit")
    plt.ylabel("Frequency")
    plt.title("Test Set: Predicted Probability Distribution")
    save_plot(os.path.join(OUTPUT_DIR, "test_prob_hist.png"))

    plt.figure(figsize=(8, 5))
    plt.hist(test_probs[y_test == 0], bins=50, alpha=0.7, label="true licit")
    plt.hist(test_probs[y_test == 1], bins=50, alpha=0.7, label="true illicit")
    plt.xlabel("Predicted probability of illicit")
    plt.ylabel("Frequency")
    plt.title("Test Set: Probability Distribution by True Class")
    plt.legend()
    save_plot(os.path.join(OUTPUT_DIR, "test_prob_by_class_hist.png"))

    fpr, tpr, _ = roc_curve(y_test, test_probs)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve on Test Set")
    plt.legend()
    save_plot(os.path.join(OUTPUT_DIR, "test_roc_curve.png"))

    precision_arr, recall_arr, _ = precision_recall_curve(y_test, test_probs)
    plt.figure(figsize=(6, 6))
    plt.plot(recall_arr, precision_arr, label=f"AP = {ap:.4f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve on Test Set")
    plt.legend()
    save_plot(os.path.join(OUTPUT_DIR, "test_pr_curve.png"))

    plt.figure(figsize=(5, 4))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.colorbar()
    plt.xticks([0, 1], ["licit", "illicit"])
    plt.yticks([0, 1], ["licit", "illicit"])
    plt.xlabel("Predicted label")
    plt.ylabel("True label")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    save_plot(os.path.join(OUTPUT_DIR, "test_confusion_matrix.png"))

    labeled_mask = ~test_set["class"].map(map_label_binary).isna()
    labeled_test_df = test_set.loc[labeled_mask].copy()
    labeled_test_df["y_true"] = labeled_test_df["class"].map(map_label_binary).astype(int)
    labeled_test_df["y_prob_illicit"] = test_probs
    labeled_test_df["y_pred"] = test_preds
    labeled_test_df.to_csv(os.path.join(OUTPUT_DIR, "test_predictions.csv"), index=False)

    return {
        "metrics": metrics_df,
        "confusion_matrix": cm
    }


# ============================================================
# Main workflow
# ============================================================

def main():
    print("Using device:", DEVICE)

    data, feature_cols = load_and_merge_data(FEATURE_PATH, CLASS_PATH, EDGE_PATH)

    print("\nMerged dataset shape:", data.shape)
    print("Class distribution:")
    print(data["class"].astype(str).value_counts(dropna=False))

    train_set, val_set, test_set, train_ts, val_ts, test_ts = split_by_time(
        data,
        train_ratio=0.70,
        val_ratio=0.15,
        test_ratio=0.15
    )

    print("\nTime-step split:")
    print(f"Train time steps: {train_ts[0]} -> {train_ts[-1]} ({len(train_ts)} steps)")
    print(f"Val   time steps: {val_ts[0]} -> {val_ts[-1]} ({len(val_ts)} steps)")
    print(f"Test  time steps: {test_ts[0]} -> {test_ts[-1]} ({len(test_ts)} steps)")

    print("\nDataset sizes:")
    print("Train:", train_set.shape)
    print("Val  :", val_set.shape)
    print("Test :", test_set.shape)

    train_set.to_csv(os.path.join(OUTPUT_DIR, "train_set.csv"), index=False)
    val_set.to_csv(os.path.join(OUTPUT_DIR, "val_set.csv"), index=False)
    test_set.to_csv(os.path.join(OUTPUT_DIR, "test_set.csv"), index=False)

    train_set_filled, fill_scaler, klr_lic, klr_ill = fill_unknown_labels_in_train(
        train_set,
        feature_cols,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        margin_threshold=MARGIN_THRESHOLD
    )

    train_set_filled.to_csv(os.path.join(OUTPUT_DIR, "train_set_filled.csv"), index=False)

    print("\nFilled training set class distribution:")
    print(train_set_filled["class_filled"].astype(str).value_counts(dropna=False))

    tuning_output = tune_final_klr(train_set_filled, val_set, feature_cols)

    best_params = tuning_output["best_params"]
    best_result = tuning_output["best_result"]

    print("\nBest hyperparameters:")
    print(best_params)

    if best_result is not None and "val_auc" in best_result["history"]:
        plt.figure(figsize=(8, 5))
        plt.plot(best_result["history"]["val_auc"])
        plt.xlabel("Epoch")
        plt.ylabel("Validation AUC")
        plt.title("Best Hyperparameter Run: Validation AUC")
        save_plot(os.path.join(OUTPUT_DIR, "best_val_auc_curve.png"))

    final_output = retrain_final_model(
        X_train=tuning_output["X_train"],
        y_train=tuning_output["y_train"],
        X_val=tuning_output["X_val"],
        y_val=tuning_output["y_val"],
        best_params=best_params
    )

    final_model = final_output["model"]
    final_scaler = final_output["scaler"]

    torch.save(final_model.state_dict(), os.path.join(OUTPUT_DIR, "final_klr_state_dict.pt"))
    np.save(
        os.path.join(OUTPUT_DIR, "final_scaler.npy"),
        {
            "mean": final_scaler.mean_,
            "scale": final_scaler.scale_
        },
        allow_pickle=True
    )

    evaluate_on_test(
        model=final_model,
        scaler=final_scaler,
        test_set=test_set,
        feature_cols=feature_cols,
        final_history=final_output["history"]
    )

    print(f"\nAll outputs saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()

Using device: cuda

Merged dataset shape: (203769, 168)
Class distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Time-step split:
Train time steps: 1 -> 34 (34 steps)
Val   time steps: 35 -> 41 (7 steps)
Test  time steps: 42 -> 49 (8 steps)

Dataset sizes:
Train: (136265, 168)
Val  : (30680, 168)
Test : (36824, 168)

Training klr_lic ...
Epoch 01/20 | train_loss=0.159798
Epoch 02/20 | train_loss=0.158226
Epoch 03/20 | train_loss=0.156726
Epoch 04/20 | train_loss=0.155313
Epoch 05/20 | train_loss=0.153950
Epoch 06/20 | train_loss=0.152656
Epoch 07/20 | train_loss=0.151415
Epoch 08/20 | train_loss=0.150219
Epoch 09/20 | train_loss=0.149074
Epoch 10/20 | train_loss=0.147973
Epoch 11/20 | train_loss=0.146915
Epoch 12/20 | train_loss=0.145898
Epoch 13/20 | train_loss=0.144914
Epoch 14/20 | train_loss=0.143973
Epoch 15/20 | train_loss=0.143066
Epoch 16/20 | train_loss=0.142188
Epoch 17/20 | train_loss=0.141349
Epoch 18/20 | train_loss=0.14053